# OpenPlaque — Plaque + Inflammation Publication Visualization v1

Clean **Runtime → Run all** notebook.

This notebook regenerates the consolidated plaque/inflammation endpoint from existing Drive caches and then produces publication-style visualizations:

- stacked plaque composition for LAD, RCA, LCX and LM;
- mean direct PCAT attenuation for RCA, LAD and LCX;
- fat-voxel-weighted local PCAT attenuation-band decomposition;
- longitudinal PCAT profiles;
- coverage-aware longitudinal PCAT heatmap;
- RCA radial PCAT gradient.

Every figure is displayed inline and saved as PNG. The full output directory, including endpoint tables, figures, CSVs, JSON and HTML report, is exported to one ZIP.

**Research boundaries:** OpenPlaque plaque volumes are research best-estimate proxies, not Cleerly outputs. Direct PCAT attenuation is not proprietary Caristo FAI-Score. LM inflammation is not standardized. The PCAT band chart classifies local 1-mm mean-HU bins weighted by their measured fat-voxel counts; it is not a voxel-level histologic decomposition.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Paths and cache controls — immediately after Drive mount
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/OpenPlaque")
OUT = DRIVE_ROOT / "Plaque_Inflammation_Publication_Visualization_v1"

REQUIRED = [
    DRIVE_ROOT / "UCLA_Plaque_Type_Estimates" / "best_estimate_plaque_types_by_artery.csv",
    DRIVE_ROOT / "RCA_Plaque_PCAT_Research_Lock_v1" / "RCA_locked_research_plaque_PCAT_profile_10_50.csv",
    DRIVE_ROOT / "LAD_Source_Space_PCAT_Feasibility_v1" / "LAD_PCAT_segment_summary.csv",
    DRIVE_ROOT / "LAD_Source_Space_PCAT_Feasibility_v1" / "frozen_LAD_PCAT_longitudinal.csv",
    DRIVE_ROOT / "LCX_OM_Source_Space_Composition_PCAT_Feasibility_v1" / "LCX_OM_PCAT_primary_summary.csv",
    DRIVE_ROOT / "LCX_OM_Source_Space_Composition_PCAT_Feasibility_v1" / "C6_PCAT_longitudinal_primary.csv",
    DRIVE_ROOT / "LCX_OM_Source_Space_Composition_PCAT_Feasibility_v1" / "C7_PCAT_longitudinal_primary.csv",
    DRIVE_ROOT / "Combined_TPV_PCAT_All_Metrics_v2" / "pcat_canonical_radial_v2.csv",
]

missing = [str(p) for p in REQUIRED if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing required cached inputs:\n" + "\n".join(missing))

OUT.mkdir(parents=True, exist_ok=True)
print("Output:", OUT)
print("All required cached evidence inputs found.")

In [ ]:
import shutil, sys, subprocess, json
from pathlib import Path

OPENPLAQUE_PIN = "a845bf993e5863f9abf1a8558dfac7379bd21775"
OPENPLAQUE_BRANCH = "plaque-inflammation-best-estimates-from-main"

if Path("/content/OpenPlaque").exists():
    shutil.rmtree("/content/OpenPlaque")

!git clone -q --branch {OPENPLAQUE_BRANCH} https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git -C /content/OpenPlaque checkout -q {OPENPLAQUE_PIN}

%pip install -q /content/OpenPlaque

for name in list(sys.modules):
    if name == "openplaque" or name.startswith("openplaque."):
        del sys.modules[name]

actual = subprocess.check_output(
    ["git","-C","/content/OpenPlaque","rev-parse","HEAD"], text=True
).strip()
print("OpenPlaque pin:", actual)
assert actual == OPENPLAQUE_PIN

In [ ]:
# Synthetic/unit tests before study-data visualization
from openplaque.plaque_inflammation_publication_visualization_v1 import synthetic_self_test

print(synthetic_self_test())
!cd /content/OpenPlaque && pytest -q   tests/test_plaque_inflammation_best_estimates_v1.py   tests/test_plaque_inflammation_publication_visualization_v1.py

In [ ]:
# Run the full endpoint fusion + visualization/export pipeline
from openplaque.plaque_inflammation_publication_visualization_v1 import run

summary = run(
    drive_root=str(DRIVE_ROOT),
    output_dir=str(OUT),
)

print(json.dumps(summary, indent=2))

In [ ]:
# Quantitative tables used by the figures
import pandas as pd
from IPython.display import display

endpoint = OUT / "endpoint"

plaque = pd.read_csv(endpoint / "plaque_best_estimates_by_vessel.csv")
aggregate = pd.read_csv(endpoint / "major_vessel_aggregate.csv")
inflammation = pd.read_csv(endpoint / "inflammation_best_estimates_by_vessel.csv")
bands = pd.read_csv(OUT / "pcat_local_mean_band_decomposition.csv")
long_bins = pd.read_csv(OUT / "pcat_longitudinal_bins_used.csv")

print("Plaque best estimates")
display(plaque[[
    "vessel",
    "tpv_best_estimate_mm3",
    "tpv_strict_or_known_lower_mm3",
    "tpv_candidate_envelope_upper_mm3",
    "ncpv_best_estimate_mm3",
    "lap_best_estimate_mm3",
    "calcified_plaque_volume_mm3",
    "confirm2_tpv_stage",
    "absolute_volume_confidence",
]])

print("\nMajor-vessel aggregate")
display(aggregate)

print("\nDirect PCAT / inflammation summary")
display(inflammation[[
    "vessel",
    "pcat_mean_hu_best_estimate",
    "pcat_segment_length_mm",
    "segment_coverage_fraction",
    "caristo_comparability",
    "fai_score",
    "confidence",
]])

print("\nPCAT local-mean attenuation bands")
display(bands)

In [ ]:
# Display every exported figure inline
from IPython.display import Image, display

FIGURES = [
    "01_plaque_composition_publication_style.png",
    "02_inflammation_mean_pcat_bar.png",
    "03_inflammation_attenuation_band_decomposition.png",
    "04_inflammation_longitudinal_profiles.png",
    "05_inflammation_longitudinal_heatmap.png",
    "06_rca_radial_pcat_gradient.png",
]

for name in FIGURES:
    p = OUT / name
    if not p.is_file():
        raise FileNotFoundError(p)
    print("\n" + name)
    display(Image(filename=str(p), width=1200))

In [ ]:
# Display the complete HTML report
from IPython.display import HTML, display

report = OUT / "OPENPLAQUE_PLAQUE_INFLAMMATION_PUBLICATION_VISUALIZATION_V1_REPORT.html"
display(HTML(report.read_text()))

In [ ]:
# Verify ZIP and all deliverables
ZIP_PATH = OUT / "OPENPLAQUE_PLAQUE_INFLAMMATION_PUBLICATION_VISUALIZATION_V1_RESULTS.zip"

expected = [
    "run_state.json",
    "summary.json",
    "pcat_longitudinal_bins_used.csv",
    "pcat_local_mean_band_decomposition.csv",
    "rca_radial_pcat_profile_used.csv",
    "01_plaque_composition_publication_style.png",
    "02_inflammation_mean_pcat_bar.png",
    "03_inflammation_attenuation_band_decomposition.png",
    "04_inflammation_longitudinal_profiles.png",
    "05_inflammation_longitudinal_heatmap.png",
    "06_rca_radial_pcat_gradient.png",
    "OPENPLAQUE_PLAQUE_INFLAMMATION_PUBLICATION_VISUALIZATION_V1_REPORT.html",
    "OPENPLAQUE_PLAQUE_INFLAMMATION_PUBLICATION_VISUALIZATION_V1_RESULTS.zip",
]

missing = [x for x in expected if not (OUT / x).exists()]
if missing:
    raise RuntimeError("Missing outputs: " + str(missing))

state = json.loads((OUT / "run_state.json").read_text())
if state.get("status") != "COMPLETE":
    raise RuntimeError("Run state not COMPLETE: " + str(state))

print("COMPLETE")
print("ZIP:", ZIP_PATH)
print("ZIP size (MB):", ZIP_PATH.stat().st_size / 1e6)
print("\nOutput files:")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(p.relative_to(OUT))